# Gradient boosting for regression: the algorithm, step by step
Three startups from the 50 Startups data (spend and profit in thousands). We run each step of the gradient boosting
algorithm by hand and check it against scikit-learn's `GradientBoostingRegressor`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, export_text

df = pd.read_csv("data/startups3.csv")
X, y = df[["rd", "admin", "marketing"]].values, df["profit"].values
df

## Step 1: the best constant is the mean

In [ ]:
# try many constants and keep the one with the smallest loss 1/2 * sum (y - gamma)^2
gammas = np.linspace(80, 200, 120001)
loss = 0.5 * ((y[:, None] - gammas[None, :]) ** 2).sum(axis=0)
print("best constant by search:", gammas[loss.argmin()].round(2), " mean:", y.mean().round(2))
df["F0"] = y.mean()

## Step 2(a): pseudo-residuals = minus the derivative of the loss

In [ ]:
# the derivative of 1/2 (y - F)^2 with respect to F is -(y - F); minus that is y - F
F0 = df["F0"].values
eps = 1e-6
numeric = -(0.5 * (y - (F0 + eps)) ** 2 - 0.5 * (y - (F0 - eps)) ** 2) / (2 * eps)   # numerical derivative
df["r1"] = y - F0
print(numeric.round(4), df["r1"].values.round(4))

## Step 2(b): a regression tree on the residuals

In [ ]:
# depth 1 because there are only three rows
dt1 = DecisionTreeRegressor(max_depth=1, random_state=0).fit(X, df["r1"])
print(export_text(dt1, feature_names=["rd", "admin", "marketing"]))
df["region"] = dt1.apply(X)          # which leaf (terminal region) each row falls in
df

## Step 2(c): the value of each terminal region

In [ ]:
# for squared error, the best value of a region is the mean of its residuals
gamma = df.groupby("region")["r1"].mean()
print(gamma)

## Step 2(d): update the model

In [ ]:
for lr in (1.0, 0.1):
    print(lr, (F0 + lr * df["region"].map(gamma)).values.round(3))

## Check against scikit-learn

In [ ]:
for lr in (1.0, 0.1):
    gbr = GradientBoostingRegressor(loss="squared_error", n_estimators=1, learning_rate=lr,
                                    max_depth=1, random_state=0).fit(X, y)
    print(lr, gbr.predict(X).round(3))

## Extra: absolute error changes step 1 and step 2(c)

In [ ]:
# with loss |y - F| the best constant is the median, the pseudo-residuals are the signs of y - F,
# and each leaf's value is the median of the actual residuals in that leaf
gbr = GradientBoostingRegressor(loss="absolute_error", n_estimators=1, learning_rate=1.0,
                                max_depth=1, random_state=0).fit(X, y)
print("F0:", np.median(y), " pseudo-residuals:", np.sign(y - np.median(y)))
print("leaf values:", np.unique(gbr.estimators_[0, 0].tree_.value.ravel()[1:]))
print("predictions:", gbr.predict(X))

In [ ]:
# the four regression losses of GradientBoostingRegressor; old names such as "ls" are rejected
for loss in ["squared_error", "absolute_error", "huber", "quantile", "ls"]:
    try:
        GradientBoostingRegressor(loss=loss, n_estimators=5).fit(X, y)
        print(loss, "works")
    except ValueError as e:
        print(loss, "-> ValueError:", str(e)[:80])